### Baseline

В качестве базовой модели использована линейная регрессия на простых числовых признаках без кодирования категориальных фич. Полученные значения RMSE/MAE/MAPE и R² служат для сравнения с более сложными моделями.

In [1]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.modeling import (
    run_baseline_pipeline,
    load_splits,
    prepare_baseline_data,
    get_baseline_features,
)

In [2]:
baseline_results = run_baseline_pipeline()
baseline_results

{'model': LinearRegression(),
 'model_path': WindowsPath('K:/HSE/hseml-group-project-pe3ricia/models/baseline_linear_regression.pkl'),
 'valid_metrics': {'RMSE': 1433031.9916384702,
  'MAE': 1242797.6562545518,
  'MAPE': np.float64(230.555772890018),
  'R2': -0.0030927931092421534},
 'test_metrics': {'RMSE': 1423656.1892574027,
  'MAE': 1232537.5610119202,
  'MAPE': np.float64(199.33933424737856),
  'R2': -0.0017515808303716351},
 'feature_cols': ['Money Laundering Risk Score',
  'Shell Companies Involved',
  'transaction_year',
  'transaction_month',
  'transaction_dayofweek',
  'transaction_hour',
  'is_illegal',
  'is_reported']}

In [27]:
import pandas as pd

experiments = []

experiments.append(
    {
        "model_name": "LinearRegression_baseline",
        "params": "default; numeric features only",
        "split": "validation",
        **baseline_results["valid_metrics"],
    }
)

experiments.append(
    {
        "model_name": "LinearRegression_baseline",
        "params": "default; numeric features only",
        "split": "test",
        **baseline_results["test_metrics"],
    }
)

experiments_df = pd.DataFrame(experiments)
experiments_df

,model_name,params,split,RMSE,MAE,MAPE,R2
0,LinearRegression_baseline,default; numeric features only,validation,1.433032e+06,1.242798e+06,230.555773,-0.003093
1,LinearRegression_baseline,default; numeric features only,test,1.423656e+06,1.232538e+06,199.339334,-0.001752


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [5]:
RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

In [6]:
DATA_DIR = Path("../data/processed")

train_df = pd.read_csv(DATA_DIR / "train.csv")
valid_df = pd.read_csv(DATA_DIR / "valid.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test :", test_df.shape)

train: (7000, 17)
valid: (1500, 17)
test : (1500, 17)


In [7]:
train_df.head()

,Country,Transaction Type,Industry,Destination Country,Money Laundering Risk Score,Shell Companies Involved,Tax Haven Country,transaction_year,transaction_month,transaction_dayofweek,transaction_hour,is_illegal,is_reported,has_tax_haven,log_amount,Financial Institution_grouped,Amount (USD)
0,Switzerland,Cryptocurrency,Luxury Goods,South Africa,10,3,Panama,2013,7,5,2,1,1,1,13.623577,OTHER,8.253611e+05
1,India,Offshore Transfer,Oil & Gas,Russia,10,9,Singapore,2013,9,3,16,1,0,1,13.222104,OTHER,5.524413e+05
2,USA,Property Purchase,Luxury Goods,Russia,5,1,Cayman Islands,2013,9,2,10,0,1,1,14.945248,OTHER,3.094842e+06
3,Switzerland,Stocks Transfer,Real Estate,Russia,3,0,Luxembourg,2014,1,6,3,1,0,1,14.746832,OTHER,2.537859e+06
4,China,Cash Withdrawal,Construction,South Africa,1,3,Singapore,2013,6,2,23,1,0,1,14.920456,OTHER,3.019059e+06


In [8]:
target_col = "Amount (USD)"

numeric_features = [
    "Money Laundering Risk Score",
    "Shell Companies Involved",
    "transaction_year",
    "transaction_month",
    "transaction_dayofweek",
    "transaction_hour",
    "is_illegal",
    "is_reported",
]

categorical_features = [
    "Country",
    "Transaction Type",
    "Industry",
    "Destination Country",
    "Tax Haven Country",
    "Financial Institution_grouped",
]

In [10]:
def mean_absolute_percentage_error(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE": mape,
        "R2": r2,
    }

In [12]:
# Универсалььная функция для оценки модели
def evaluate_model(
    model,
    X_train,
    y_train,
    X_valid,
    y_valid,
    X_test,
    y_test,
    model_name,
    feature_set,
    params_description="default",
):
    model.fit(X_train, y_train)

    valid_pred = model.predict(X_valid)
    test_pred = model.predict(X_test)

    valid_metrics = calculate_regression_metrics(y_valid, valid_pred)
    test_metrics = calculate_regression_metrics(y_test, test_pred)

    valid_row = {
        "model_name": model_name,
        "feature_set": feature_set,
        "params": params_description,
        "split": "validation",
        **valid_metrics,
    }

    test_row = {
        "model_name": model_name,
        "feature_set": feature_set,
        "params": params_description,
        "split": "test",
        **test_metrics,
    }

    return valid_row, test_row, valid_pred, test_pred

In [13]:
target_col = "Amount (USD)"

numeric_features = [
    "Money Laundering Risk Score",
    "Shell Companies Involved",
    "transaction_year",
    "transaction_month",
    "transaction_dayofweek",
    "transaction_hour",
    "is_illegal",
    "is_reported",
]

categorical_features = [
    "Country",
    "Transaction Type",
    "Industry",
    "Destination Country",
    "Tax Haven Country",
    "Financial Institution_grouped",
]

all_features = numeric_features + categorical_features
all_features

['Money Laundering Risk Score',
 'Shell Companies Involved',
 'transaction_year',
 'transaction_month',
 'transaction_dayofweek',
 'transaction_hour',
 'is_illegal',
 'is_reported',
 'Country',
 'Transaction Type',
 'Industry',
 'Destination Country',
 'Tax Haven Country',
 'Financial Institution_grouped']

In [14]:
X_train_raw = train_df[all_features].copy()
y_train = train_df[target_col].copy()

X_valid_raw = valid_df[all_features].copy()
y_valid = valid_df[target_col].copy()

X_test_raw = test_df[all_features].copy()
y_test = test_df[target_col].copy()

print("X_train_raw:", X_train_raw.shape)
print("X_valid_raw:", X_valid_raw.shape)
print("X_test_raw :", X_test_raw.shape)

X_train_raw: (7000, 14)
X_valid_raw: (1500, 14)
X_test_raw : (1500, 14)


In [15]:
X_train_full = pd.get_dummies(X_train_raw, columns=categorical_features, drop_first=False)
X_valid_full = pd.get_dummies(X_valid_raw, columns=categorical_features, drop_first=False)
X_test_full = pd.get_dummies(X_test_raw, columns=categorical_features, drop_first=False)

In [16]:
# Выравнивание
X_valid_full = X_valid_full.reindex(columns=X_train_full.columns, fill_value=0)
X_test_full = X_test_full.reindex(columns=X_train_full.columns, fill_value=0)

In [17]:
print("X_train_full:", X_train_full.shape)
print("X_valid_full:", X_valid_full.shape)
print("X_test_full :", X_test_full.shape)

X_train_full: (7000, 67)
X_valid_full: (1500, 67)
X_test_full : (1500, 67)


In [18]:
X_train_full.head()

,Money Laundering Risk Score,Shell Companies Involved,transaction_year,transaction_month,transaction_dayofweek,transaction_hour,is_illegal,is_reported,Country_Brazil,Country_China,...,Financial Institution_grouped_Bank_319,Financial Institution_grouped_Bank_321,Financial Institution_grouped_Bank_328,Financial Institution_grouped_Bank_349,Financial Institution_grouped_Bank_41,Financial Institution_grouped_Bank_428,Financial Institution_grouped_Bank_434,Financial Institution_grouped_Bank_438,Financial Institution_grouped_Bank_81,Financial Institution_grouped_OTHER
0,10,3,2013,7,5,2,1,1,False,False,...,False,False,False,False,False,False,False,False,False,True
1,10,9,2013,9,3,16,1,0,False,False,...,False,False,False,False,False,False,False,False,False,True
2,5,1,2013,9,2,10,0,1,False,False,...,False,False,False,False,False,False,False,False,False,True
3,3,0,2014,1,6,3,1,0,False,False,...,False,False,False,False,False,False,False,False,False,True
4,1,3,2013,6,2,23,1,0,False,True,...,False,False,False,False,False,False,False,False,False,True


### Кодирование категориальных признаков

Поскольку большая часть информативности датасета содержится в категориальных признаках (страны, тип транзакции, индустрия, офшорная юрисдикция, финансовый институт), для дальнейших экспериментов был сформирован полный набор признаков с one-hot кодированием. Для этого использовался `pd.get_dummies`, после чего наборы train, validation и test были приведены к единому пространству признаков через выравнивание колонок.

In [30]:
from sklearn.linear_model import LinearRegression

linreg_full = LinearRegression()

valid_row_full, test_row_full, valid_pred_full, test_pred_full = evaluate_model(
    model=linreg_full,
    X_train=X_train_full,
    y_train=y_train,
    X_valid=X_valid_full,
    y_valid=y_valid,
    X_test=X_test_full,
    y_test=y_test,
    model_name="LinearRegression_full_ohe",
    feature_set="numeric_plus_ohe",
    params_description="default",
)

In [28]:
experiments_df = pd.concat(
    [
        experiments_df,
        pd.DataFrame([valid_row_full, test_row_full])
    ],
    ignore_index=True
)

experiments_df

,model_name,params,split,RMSE,MAE,MAPE,R2,feature_set
0,LinearRegression_baseline,default; numeric features only,validation,1.433032e+06,1.242798e+06,230.555773,-0.003093,NaN
1,LinearRegression_baseline,default; numeric features only,test,1.423656e+06,1.232538e+06,199.339334,-0.001752,NaN
2,LinearRegression_full_ohe,default,validation,1.443479e+06,1.250130e+06,231.826935,-0.017772,numeric_plus_ohe
3,LinearRegression_full_ohe,default,test,1.429288e+06,1.235304e+06,200.156635,-0.009693,numeric_plus_ohe


In [31]:
from sklearn.ensemble import RandomForestRegressor

In [42]:
rf_full = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=RANDOM_STATE,
    n_jobs=3
)

In [49]:
valid_row_rf, test_row_rf, valid_pred_rf, test_pred_rf = evaluate_model(
    model=rf_full,
    X_train=X_train_full,
    y_train=y_train,
    X_valid=X_valid_full,
    y_valid=y_valid,
    X_test=X_test_full,
    y_test=y_test,
    model_name="RandomForest_full_ohe",
    feature_set="numeric_plus_ohe",
    params_description="n_estimators=300,max_depth=12,min_samples_split=10,min_samples_leaf=5",
)

In [50]:
experiments_df = pd.concat(
    [
        experiments_df,
        pd.DataFrame([valid_row_rf, test_row_rf])
    ],
    ignore_index=True
)

experiments_df.tail()

,model_name,params,split,RMSE,MAE,MAPE,R2,feature_set
1,LinearRegression_baseline,default; numeric features only,test,1.423656e+06,1.232538e+06,199.339334,-0.001752,NaN
2,LinearRegression_full_ohe,default,validation,1.443479e+06,1.250130e+06,231.826935,-0.017772,numeric_plus_ohe
3,LinearRegression_full_ohe,default,test,1.429288e+06,1.235304e+06,200.156635,-0.009693,numeric_plus_ohe
4,RandomForest_full_ohe,"n_estimators=300,max_depth=12,min_samples_spli...",validation,1.434509e+06,1.243530e+06,229.928602,-0.005162,numeric_plus_ohe
5,RandomForest_full_ohe,"n_estimators=300,max_depth=12,min_samples_spli...",test,1.430020e+06,1.237078e+06,200.428631,-0.010727,numeric_plus_ohe


In [35]:
print(X_train_full.shape)
print(X_valid_full.shape)
print(X_test_full.shape)

(7000, 67)
(1500, 67)
(1500, 67)


In [36]:
print((X_train_full.columns == X_valid_full.columns).all())
print((X_train_full.columns == X_test_full.columns).all())

True
True


In [37]:
print(X_train_full.isna().sum().sum())
print(X_valid_full.isna().sum().sum())
print(X_test_full.isna().sum().sum())

0
0
0


In [38]:
rf_full.fit(X_train_full, y_train)

train_pred = rf_full.predict(X_train_full)
valid_pred = rf_full.predict(X_valid_full)

print(calculate_regression_metrics(y_train, train_pred))
print(calculate_regression_metrics(y_valid, valid_pred))

{'RMSE': 1219906.8024336682, 'MAE': 1049165.5748827076, 'MAPE': np.float64(200.09974072755136), 'R2': 0.2652458271018294}
{'RMSE': 1434945.9328795501, 'MAE': 1243936.1253808101, 'MAPE': np.float64(229.97545014381825), 'R2': -0.005774021091051296}


In [39]:
print("target train:", y_train.describe())
print("target valid:", y_valid.describe())
print("target test :", y_test.describe())

target train: count    7.000000e+03
mean     2.494221e+06
std      1.423268e+06
min      1.053049e+04
25%      1.270545e+06
50%      2.487898e+06
75%      3.706783e+06
max      4.999812e+06
Name: Amount (USD), dtype: float64
target valid: count    1.500000e+03
mean     2.539488e+06
std      1.431298e+06
min      1.003180e+04
25%      1.278920e+06
50%      2.572080e+06
75%      3.794856e+06
max      4.996409e+06
Name: Amount (USD), dtype: float64
target test : count    1.500000e+03
mean     2.499597e+06
std      1.422885e+06
min      1.939589e+04
25%      1.309672e+06
50%      2.499300e+06
75%      3.690647e+06
max      4.997300e+06
Name: Amount (USD), dtype: float64


In [40]:
from sklearn.linear_model import Ridge

ridge_full = Ridge(alpha=1.0)

valid_row_ridge, test_row_ridge, valid_pred_ridge, test_pred_ridge = evaluate_model(
    model=ridge_full,
    X_train=X_train_full,
    y_train=y_train,
    X_valid=X_valid_full,
    y_valid=y_valid,
    X_test=X_test_full,
    y_test=y_test,
    model_name="Ridge_full_ohe",
    feature_set="numeric_plus_ohe",
    params_description="alpha=1.0",
)

In [41]:
X_train_full.dtypes

Money Laundering Risk Score               int64
Shell Companies Involved                  int64
transaction_year                          int64
transaction_month                         int64
transaction_dayofweek                     int64
                                          ...  
Financial Institution_grouped_Bank_428     bool
Financial Institution_grouped_Bank_434     bool
Financial Institution_grouped_Bank_438     bool
Financial Institution_grouped_Bank_81      bool
Financial Institution_grouped_OTHER        bool
Length: 67, dtype: object

In [51]:
experiments_df = pd.concat(
    [
        experiments_df,
        pd.DataFrame([valid_row_ridge, test_row_ridge])
    ],
    ignore_index=True
)

experiments_df.tail()

,model_name,params,split,RMSE,MAE,MAPE,R2,feature_set
3,LinearRegression_full_ohe,default,test,1.429288e+06,1.235304e+06,200.156635,-0.009693,numeric_plus_ohe
4,RandomForest_full_ohe,"n_estimators=300,max_depth=12,min_samples_spli...",validation,1.434509e+06,1.243530e+06,229.928602,-0.005162,numeric_plus_ohe
5,RandomForest_full_ohe,"n_estimators=300,max_depth=12,min_samples_spli...",test,1.430020e+06,1.237078e+06,200.428631,-0.010727,numeric_plus_ohe
6,Ridge_full_ohe,alpha=1.0,validation,1.443178e+06,1.249890e+06,231.761586,-0.017347,numeric_plus_ohe
7,Ridge_full_ohe,alpha=1.0,test,1.429146e+06,1.235170e+06,200.179198,-0.009493,numeric_plus_ohe


In [52]:
y_train_log = np.log1p(y_train)
y_valid_log = np.log1p(y_valid)
y_test_log = np.log1p(y_test)

In [57]:
ridge_log = Ridge(alpha=1.0)

ridge_log.fit(X_train_full, y_train_log)

valid_row_ridge_log, test_row_ridge_log, valid_pred_ridge_log, test_pred_ridge_log = evaluate_model(
    model=ridge_log,
    X_train=X_train_full,
    y_train=y_train_log,
    X_valid=X_valid_full,
    y_valid=y_valid_log,
    X_test=X_test_full,
    y_test=y_test_log,
    model_name="Ridge_log_target",
    feature_set="numeric_plus_ohe",
    params_description="alpha=1.0, target=log1p(amount)",
)

In [58]:
valid_pred_amount = np.expm1(valid_pred_ridge_log)
test_pred_amount = np.expm1(test_pred_ridge_log)

In [59]:
valid_metrics_amount = calculate_regression_metrics(y_valid, valid_pred_amount)
test_metrics_amount = calculate_regression_metrics(y_test, test_pred_amount)

valid_metrics_amount, test_metrics_amount

({'RMSE': 1590306.9367113772,
  'MAE': 1336548.0089994152,
  'MAPE': np.float64(178.47762116370413),
  'R2': -0.23535348562953007},
 {'RMSE': 1563676.312782521,
  'MAE': 1312592.6599861677,
  'MAPE': np.float64(153.36025244055546),
  'R2': -0.2084912494695994})

## Изменение постановки задачи

Ещё на этапе EDA было замечено, что между признаками и целевой переменной `Amount (USD)` наблюдается слабая связь, а распределение самой целевой переменной выглядит не вполне подходящим для устойчивого предсказания. Это уже на начальном этапе указывало на ограниченный потенциал задачи регрессии для выбранного таргета.

Дополнительно в ходе экспериментов было установлено, что обученные модели сходятся за считанные секунды, но при этом показывают качество, близкое к наивному предсказанию средним значением, а в отдельных случаях — даже хуже него. Это означает, что модели практически не извлекают полезного сигнала из имеющихся признаков для предсказания `Amount (USD)`.

Позже также выяснилось, что используемый датасет является синтетическим и изначально создавался для другой целевой переменной. Поэтому дальнейшая работа с таргетом `Amount (USD)` представляется методологически нецелесообразной.

В связи с этим постановка задачи меняется: далее будет проведён EDA для нового таргета — уровня риска (`Money Laundering Risk Score`) в файле `EDA_risk`, после чего последующий анализ и моделирование будут выполняться уже для новой целевой переменной.